In [1]:
# ============================================
# Transformer V4 - Imports & Dataset
# ============================================

import os
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

print("TensorFlow Version:", tf.__version__)

# Dataset path
DATA_PATH = "../dataset/training"

# Load dataset
X_train = np.load(os.path.join(DATA_PATH, "X_train.npy"))
y_train = np.load(os.path.join(DATA_PATH, "y_train.npy"))

X_val = np.load(os.path.join(DATA_PATH, "X_val.npy"))
y_val = np.load(os.path.join(DATA_PATH, "y_val.npy"))

X_test = np.load(os.path.join(DATA_PATH, "X_test.npy"))
y_test = np.load(os.path.join(DATA_PATH, "y_test.npy"))

print("\n✅ Dataset Loaded Successfully!")
print("Train      :", X_train.shape, y_train.shape)
print("Validation :", X_val.shape, y_val.shape)
print("Test       :", X_test.shape, y_test.shape)

# Basic info
num_classes = len(np.unique(y_train))
sequence_length = X_train.shape[1]
feature_dim = X_train.shape[2]

print("\nClasses         :", num_classes)
print("Sequence Length :", sequence_length)
print("Feature Dim     :", feature_dim)

TensorFlow Version: 2.21.0

✅ Dataset Loaded Successfully!
Train      : (7320, 154, 63) (7320,)
Validation : (630, 154, 63) (630,)
Test       : (420, 154, 63) (420,)

Classes         : 210
Sequence Length : 154
Feature Dim     : 63


In [2]:
# ============================================
# Transformer V4 - Normalize Dataset
# ============================================

mean = np.mean(X_train, axis=(0, 1), keepdims=True)
std = np.std(X_train, axis=(0, 1), keepdims=True) + 1e-6

X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std

print("✅ Normalization Complete!")

print("Train mean:", round(float(np.mean(X_train)), 6))
print("Train std :", round(float(np.std(X_train)), 6))
print("Val mean  :", round(float(np.mean(X_val)), 6))
print("Test mean :", round(float(np.mean(X_test)), 6))

✅ Normalization Complete!
Train mean: -3e-06
Train std : 0.999626
Val mean  : -0.002387
Test mean : -0.006258


In [3]:
# ============================================
# Transformer V4 Architecture
# ============================================

from tensorflow.keras import layers, Model

# ----- Transformer Encoder Block -----
def transformer_encoder(x, head_size, num_heads, ff_dim, dropout=0.25):

    attention = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=head_size,
        dropout=dropout
    )(x, x)

    x = layers.Add()([x, attention])
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    ff = layers.Dense(ff_dim, activation="gelu")(x)
    ff = layers.Dropout(dropout)(ff)
    ff = layers.Dense(x.shape[-1])(ff)

    x = layers.Add()([x, ff])
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    return x


# ----- Input -----
inputs = layers.Input(shape=(sequence_length, feature_dim))

# Project 63 features -> 128 dimensions
x = layers.Dense(128)(inputs)

# Learnable positional embeddings
positions = tf.range(start=0, limit=sequence_length, delta=1)
position_embedding = layers.Embedding(
    input_dim=sequence_length,
    output_dim=128
)(positions)

x = x + position_embedding

# ----- 2 Transformer Encoder Blocks -----
for _ in range(2):
    x = transformer_encoder(
        x,
        head_size=32,
        num_heads=4,
        ff_dim=256,
        dropout=0.25
    )

# Classification Head
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.30)(x)
x = layers.Dense(128, activation="gelu")(x)
x = layers.Dropout(0.30)(x)

outputs = layers.Dense(num_classes, activation="softmax")(x)

model = Model(inputs, outputs)

print("✅ Transformer V4 model built!")
model.summary()

✅ Transformer V4 model built!


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 154, 63)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 154, 128)  │      8,192 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 154, 128)  │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 154, 128)  │     66,048 │ add[0][0],        │
│ (MultiHeadAttentio… │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 154, 128)  │          0 │ add[0][0],        │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 154, 128)  │        256 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 154, 256)  │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 154, 256)  │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 154, 128)  │     32,896 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 154, 128)  │          0 │ layer_normalizat… │
│                     │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 154, 128)  │        256 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 154, 128)  │     66,048 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 154, 128)  │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 154, 128)  │        256 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 154, 256)  │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 154, 256)  │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 154, 128)  │     32,896 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 154, 128)  │          0 │ layer_normalizat… │
│                     │                   │            │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 154, 128)  │        256 │ add_4[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 316,754 (1.21 MB)

 Trainable params: 316,754 (1.21 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# ============================================
# Compile Transformer V4.1
# ============================================

import tensorflow as tf

optimizer = tf.keras.optimizers.Adam(
    learning_rate=5e-4
)

model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

print("✅ Transformer V4.1 compiled successfully!")

✅ Transformer V4.1 compiled successfully!


In [7]:
# ============================================
# Transformer V4.1 Training (Balanced Dataset)
# ============================================

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=10,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "../models/best_transformer_v4_1.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=32,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

print("✅ Transformer V4.1 training complete!")

Epoch 1/60
229/229 ━━━━━━━━━━━━━━━━━━━━ 0s 297ms/step - accuracy: 0.0122 - loss: 5.2078
Epoch 1: val_accuracy improved from None to 0.03492, saving model to ../models/best_transformer_v4_1.keras

Epoch 1: finished saving model to ../models/best_transformer_v4_1.keras
229/229 ━━━━━━━━━━━━━━━━━━━━ 78s 308ms/step - accuracy: 0.0122 - loss: 5.2078 - val_accuracy: 0.0349 - val_loss: 4.8767
Epoch 2/60
229/229 ━━━━━━━━━━━━━━━━━━━━ 0s 315ms/step - accuracy: 0.0454 - loss: 4.6049
Epoch 2: val_accuracy improved from 0.03492 to 0.08730, saving model to ../models/best_transformer_v4_1.keras

Epoch 2: finished saving model to ../models/best_transformer_v4_1.keras
229/229 ━━━━━━━━━━━━━━━━━━━━ 74s 324ms/step - accuracy: 0.0454 - loss: 4.6049 - val_accuracy: 0.0873 - val_loss: 4.1909
Epoch 3/60
229/229 ━━━━━━━━━━━━━━━━━━━━ 0s 340ms/step - accuracy: 0.0919 - loss: 4.0673
Epoch 3: val_accuracy improved from 0.08730 to 0.13492, saving model to ../models/best_transformer_v4_1.keras

Epoch 3: finished savi

In [9]:
# ============================================
# Evaluate Transformer V4.1
# ============================================

test_loss, test_acc = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("=" * 40)
print(f"Transformer V4.1 Test Accuracy : {test_acc*100:.2f}%")
print(f"Transformer V4.1 Test Loss     : {test_loss:.4f}")
print("=" * 40)

14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 138ms/step - accuracy: 0.8262 - loss: 0.5997
Transformer V4.1 Test Accuracy : 82.62%
Transformer V4.1 Test Loss     : 0.5997
